In [2]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Literal

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END

import sys
sys.path.append("../ingestion/python/data-enrichment-LLM")
from silver_enrichment import *
sys.path.append("../ingestion/python/src")
from database import Database



c:\Users\rolan\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


ModuleNotFoundError: No module named 'silver_enrichment'

In [4]:
"""Get data from bronze who haven't been treated"""
"""DO DISTINCT""" 

   # def get_raw_data(state: JobOfferState, db: Database) -> JobOfferState:
    #     query = """
    #         SELECT         
    #             rjo.id_job, 
    #             rjo.api_source, 
    #             rjo.job_title, 
    #             rjo.contract_type, 
    #             rjo.job_publisher, 
    #             rjo.company, 
    #             rjo.company_website, 
    #             rjo.location_raw,
    #             rjo.city,
    #             rjo.country,
    #             rjo.latitude,
    #             rjo.longitude,
    #             rjo.is_remote,
    #             rjo.offer_description,
    #             rjo.offer_url,
    #             rjo.source_platform,
    #             rjo.published_at,
    #             rjo.collected_at
    #         FROM raw.job_offer rjo
    #         LEFT JOIN analytics.job_offer ajo
    #         ON rjo.id_job = ajo.id_job
    #         WHERE ajo.id_job IS NULL
    #         ORDER BY DESC rjo.collected_at
    #         LIMIT 10;
    #     """
    #     result = db.execture(query)


'DO DISTINCT'

In [ ]:
db = Database()
query = """
    SELECT 
        rjo.id_job, 
        rjo.api_source, 
        rjo.job_title, 
        rjo.contract_type, 
        rjo.job_publisher, 
        rjo.company, 
        rjo.company_website, 
        rjo.location_raw,
        rjo.city,
        rjo.country,
        rjo.latitude,
        rjo.longitude,
        rjo.is_remote,
        rjo.offer_description,
        rjo.offer_url,
        rjo.source_platform,
        rjo.published_at,
        rjo.collected_at
    FROM raw.job_offer rjo
    WHERE rjo.company = ''
    LIMIT 10;
"""

line = db.execute(query)

In [ ]:
line

[{'id_job': 'cj_c54e2eeee7d8ae0c7ae109464f711008',
  'api_source': 'careerjet',
  'job_title': 'Environmental Remediation Project Coordinator (Santiago)',
  'contract_type': None,
  'job_publisher': None,
  'company': '',
  'company_website': None,
  'location_raw': 'Santiago, Región Metropolitana',
  'city': 'Santiago, Región Metropolitana',
  'country': 'CL',
  'latitude': None,
  'longitude': None,
  'is_remote': None,
  'offer_description': 'Description  Our Environmental business in Santiago is looking for an Environmental <b>Engineer</b>, Environmental Scientist... may include the collection and synthesizing of samples and environmental <b>data</b>, and recommend action based on <b>data</b> derived',
  'offer_url': 'https://jobviewtrack.com/v2/3EFs_b6aZoD9Cop_7XWz3adBP9We3KKxbsX7Bhu7OR4EK4jTDotILJkCJ3nKoSVIqfGIh3oNT-gs067ailjg8qU3Tux57cnFKhKWIQvZqPgOBRO1ByKEmdK1pTaGyzoZQL6j84REdgw4A0DyFlb7jXSnWew5pU9H-noLjVsecxGWkxnHuJ7QKHn1dEvUCE9KdeJhZid3mVbDv0dpETAmEdzqRdyhjSvY2sGG3oKKvEQ',
  

In [ ]:
llm = LLM()
extract_company = Extract(
    llm=llm.llama4_smart,
    task='Find the name of the company who recruit for this job offer based on the title and the description of the job.',
    output_key='company_name',
    schema=CompanyOutput,
    fields=['job_title', 'offer_description']
)

builder = StateGraph(JobOfferState)
builder.add_node("extract_company", extract_company)
builder.add_edge(START, "extract_company")
builder.add_edge("extract_company", END)

graph = builder.compile()

NameError: name 'CompanyOutput' is not defined

In [ ]:
for i in range(len(line)):
    real_state = line[i]
    result = graph.invoke(real_state)
    print(f"id={result['id_job']}, company_name={result['company_name']}")
    

id=cj_c54e2eeee7d8ae0c7ae109464f711008, company_name=null
id=cj_59cf47c22d3120c4c544a4cabfc915ca, company_name=Cencosud Media
id=cj_786203f7c225b3036202b80975d2206b, company_name=null
id=cj_fef20ce3a0164e2b99fa0c3c3fdd577a, company_name=Thoughtworks
id=cj_19971103e452c44dd99b69afe7026187, company_name=null
id=cj_fdd47bba11be1f542a3df663db7324ee, company_name=Experian
id=cj_48afdfe1187458d5f203eda5f2c2609f, company_name=Experian plc
id=cj_7600d1af4cb8b5b7595d6fe41bdc9bfd, company_name=Empresa confidencial
id=cj_53f06653ca5c39aedf775869b0ca821e, company_name=null
id=cj_1b243354ebeccfaff9f081c9517b9f5e, company_name=null


```python
llama3
id=cj_c54e2eeee7d8ae0c7ae109464f711008, company_name=null 
id=cj_59cf47c22d3120c4c544a4cabfc915ca, company_name=Cencosud 
id=cj_786203f7c225b3036202b80975d2206b, company_name=Microsoft 
id=cj_fef20ce3a0164e2b99fa0c3c3fdd577a, company_name=Thoughtworks 
id=cj_19971103e452c44dd99b69afe7026187, company_name=null 
id=cj_fdd47bba11be1f542a3df663db7324ee, company_name=Experian 
id=cj_48afdfe1187458d5f203eda5f2c2609f, company_name=Experian 
id=cj_7600d1af4cb8b5b7595d6fe41bdc9bfd, company_name=Empresa confidencial 
id=cj_53f06653ca5c39aedf775869b0ca821e, company_name=AECOM 
id=cj_1b243354ebeccfaff9f081c9517b9f5e, company_name=None

llama4 (GRAND VAINQUEUR)
id=cj_c54e2eeee7d8ae0c7ae109464f711008, company_name=null
id=cj_59cf47c22d3120c4c544a4cabfc915ca, company_name=Cencosud Media
id=cj_786203f7c225b3036202b80975d2206b, company_name=null
id=cj_fef20ce3a0164e2b99fa0c3c3fdd577a, company_name=Thoughtworks
id=cj_19971103e452c44dd99b69afe7026187, company_name=null
id=cj_fdd47bba11be1f542a3df663db7324ee, company_name=Experian
id=cj_48afdfe1187458d5f203eda5f2c2609f, company_name=Experian plc
id=cj_7600d1af4cb8b5b7595d6fe41bdc9bfd, company_name=Empresa confidencial
id=cj_53f06653ca5c39aedf775869b0ca821e, company_name=null
id=cj_1b243354ebeccfaff9f081c9517b9f5e, company_name=null
```

In [ ]:
result

{'id_job': 'cj_1b243354ebeccfaff9f081c9517b9f5e',
 'api_source': 'careerjet',
 'job_title': 'Ingeniero Senior Back-end',
 'offer_description': 'behaviors--dark-mode-switch#select" <b>data</b>-behaviors--dark-mode-switch-target="option" <b>data</b>-preference="system" role...="menuitemradio" type="button"&gt; System  behaviors--dark-mode-switch#select" <b>data</b>-behaviors--dark-mode-switch-target="option" <b>data</b>',
 'contract_type': None,
 'is_remote': None,
 'job_publisher': None,
 'location_raw': 'Santiago, Región Metropolitana',
 'offer_url': 'https://jobviewtrack.com/v2/pZL0R2VaiXsUOurktAjyJJQ7f2pzcvkcMZDRJS9CCI5P5Li6kmswjX95I-Hbn6KvxyKLxm1n8gPiirUr7kqnQUrREwyV3SDQs3jPsTHv4BnYH3yWmDCNKC3UbJ9PG3NUsBzBLOHEz39r6uoqtajKCXgwiPpO4KEjFi6cOcmz_TEcLf8e_F5KxoKBsHrZa5BSkltbWziRbaj6O7tXOuaP97WBvwRM0Kfsr3BFDtAfRHY',
 'source_platform': None,
 'published_at': '2026-06-07 06:00:15+00',
 'collected_at': '2026-06-09 22:45:48.488771+00',
 'company_name': 'null',
 'company_website': None,
 'city

In [ ]:
from rapidfuzz import fuzz, process

c = 'Le chat noir'
r = 'caht noir'

score = fuzz.partial_ratio(r, None)
score

0

**TEST Language if LLM is needed**

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from pydantic import BaseModel, Field
from typing import Literal
from langdetect import detect
import langid
from lingua import Language, LanguageDetectorBuilder
import sys
sys.path.append("../ingestion/python/src")
from database import Database
sys.path.append("../ingestion/python/data-enrichment-LLM")
from silver_enrichment import *

class LanguageOutput(BaseModel):
    offer_language: Literal["es", "en", "fr", "pt", "unknown"] = Field(
        description="Language in which the job offer is written. Infer from the text content only, not from job requirements or technologies mentioned."
    )
llm = LLM()
extract_language_llm = Extract(
    llm=llm.llama4_smart,
    task="Detect the language in which this job offer is written.",
    output_key="offer_language_llm",
    schema=LanguageOutput,
    fields=["job_title", "offer_description"]
)




def detect_language(text: str) -> str:
    try:
        return detect(text)
    except:
        return 'unknow'

detector = LanguageDetectorBuilder.from_all_languages().build()
db = Database()
query = """
    SELECT job_title, offer_description
    FROM raw.job_offer
    WHERE job_title IS NOT NULL
    AND offer_description IS NOT NULL
    LIMIT 10;
"""
result = db.execute(query=query)

In [ ]:
for item in result:
    text = f"{item['job_title']} {item['offer_description']}"
    llm_result = extract_language_llm({
        "job_title": item["job_title"],
        "offer_description": item["offer_description"]
    })
    print("=========================================")
    print(text)
    print(f"langdetect -> {detect_language(text)}")
    print(f"langig -> {langid.classify(text)}")
    print(f"lingua -> {detector.detect_language_of(text)}")
    print(f"LLM -> {llm_result.get('offer_language_llm')}")
    print("=========================================")
    print('\n')



In [1]:
LANGUAGE_MAP = {
    Language.ENGLISH: "en",
    Language.FRENCH: "fr",
    Language.SPANISH: "es",
    Language.PORTUGUESE: "pt",
}

for item in result:
    text = f"{item['job_title']}"
    
    llm_result = extract_language_llm({
        "job_title": item["job_title"],
        "offer_description": item["offer_description"]
    })

    results = {
        "langdetect": detect_language(text),
        "langid":     langid.classify(text)[0],
        "lingua":     LANGUAGE_MAP.get(detector.detect_language_of(text), "unknown"),
        "llm":        llm_result.get("offer_language_llm")
    }

    # Détecte si un résultat diverge des autres
    values = list(results.values())
    majority = max(set(values), key=values.count)
    divergents = {k: v for k, v in results.items() if v != majority}

    print("=" * 60)
    print(f"titre : {item['job_title'][:60]}")
    for tool, lang in results.items():
        marker = "❌" if tool in divergents else "✅"
        print(f"  {marker} {tool:<12} → {lang}")

    if divergents:
        print(f"\n  ⚠️  divergence détectée : {divergents}")
        
        # Itère uniquement sur la description avec les outils sans LLM
        desc = item["offer_description"]
        desc_results = {
            "langdetect": detect_language(desc),
            "langid":     langid.classify(desc)[0],
            "lingua":     LANGUAGE_MAP.get(detector.detect_language_of(desc), "unknown"),
        }
        
        desc_values = list(desc_results.values())
        desc_majority = max(set(desc_values), key=desc_values.count)
        
        print(f"\n  🔍 Vérification sur description seule :")
        for tool, lang in desc_results.items():
            marker = "✅" if lang == desc_majority else "❌"
            print(f"     {marker} {tool:<12} → {lang}")
        
        # Le LLM a-t-il raison ?
        llm_lang = results["llm"]
        if llm_lang == desc_majority:
            print(f"\n  ✅ LLM confirmé par la description ({llm_lang})")
        else:
            print(f"\n  ❌ LLM diverge de la description (LLM={llm_lang}, desc={desc_majority})")
    print("=" * 60)

NameError: name 'Language' is not defined

In [ ]:
result

[{'job_title': 'Analytics Engineer SSR (Buenos Aires)',
  'offer_description': ' o más años de experiencia en roles de <b>Data</b> <b>Engineer</b>, Analytics <b>Engineer</b> o BI Developer. (excluyente)  - Experiencia previa en tecnologías Cloud... y transformar datos para optimizar su consumo analítico, aplicando buenas prácticas de <b>data</b> warehousing.  - Desarrollar dashboards'},
 {'job_title': 'Analytics Engineer SSR (Buenos Aires)',
  'offer_description': ' o más años de experiencia en roles de <b>Data</b> <b>Engineer</b>, Analytics <b>Engineer</b> o BI Developer. (excluyente)   - Experiencia previa en tecnologías Cloud... y transformar datos para optimizar su consumo analítico, aplicando buenas prácticas de <b>data</b> warehousing.   - Desarrollar dashboards'},
 {'job_title': 'Network Architecture & Discovery Lead',
  'offer_description': ' baselines. The position leads a specialized team of ETL <b>data</b> analysts, an infrastructure simulation <b>engineer</b>, and network..

In [1]:
import sys
sys.path.append("../ingestion/python/src")
from APIendpoint import PlacesAPI
sys.path.append("../ingestion/python/data-enrichment-LLM")
from silver_enrichment import *

places_api = PlacesAPI()
find_location = FindLocation(geo_api=places_api)

test_state = {
    "company_name": "Accenture",
    "location_raw": "Santiago, Región Metropolitana",
    "city": "Santiago"
}
result = find_location(test_state)
print(result)

c:\APPS\Python312\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Loading formatted geocoded file...
{'company_name': 'Accenture', 'company_primary_type': 'consultant', 'company_website': 'https://www.accenture.com/cl-es/about/company-index?c=acn_glb_locationmarketiyext_14234619&n=otc_0525', 'address': 'Torre Apoquindo - Av. Apoquindo 5550, Piso 14, 7560943 Las Condes, Región Metropolitana, Chile', 'lat': -33.409603499999996, 'lon': -70.5714267, 'phone': '+56 2 2337 7100', 'business_status': 'OPERATIONAL', 'rating': 4.7, 'rating_count': 67, 'city': 'Santiago Metropolitan', 'country': 'CL'}


In [2]:
result

{'company_name': 'Accenture',
 'company_primary_type': 'consultant',
 'company_website': 'https://www.accenture.com/cl-es/about/company-index?c=acn_glb_locationmarketiyext_14234619&n=otc_0525',
 'address': 'Torre Apoquindo - Av. Apoquindo 5550, Piso 14, 7560943 Las Condes, Región Metropolitana, Chile',
 'lat': -33.409603499999996,
 'lon': -70.5714267,
 'phone': '+56 2 2337 7100',
 'business_status': 'OPERATIONAL',
 'rating': 4.7,
 'rating_count': 67,
 'city': 'Santiago Metropolitan',
 'country': 'CL'}

In [3]:
import reverse_geocoder

result = {'company_name': 'Accenture',
 'company_primary_type': 'consultant',
 'company_website': 'https://www.accenture.com/cl-es/about/company-index?c=acn_glb_locationmarketiyext_14234619&n=otc_0525',
 'address': 'Torre Apoquindo - Av. Apoquindo 5550, Piso 14, 7560943 Las Condes, Región Metropolitana, Chile',
 'lat': -33.409603499999996,
 'lon': -70.5714267,
 'phone': '+56 2 2337 7100',
 'business_status': 'OPERATIONAL',
 'rating': 4.7,
 'rating_count': 67}


a = reverse_geocoder.search((result['lat'], result['lon']))

Loading formatted geocoded file...


In [11]:
type(a)
a

[{'lat': '-33.46069',
  'lon': '-70.58024',
  'name': 'Villa Presidente Frei, Nunoa, Santiago, Chile',
  'admin1': 'Santiago Metropolitan',
  'admin2': 'Provincia de Santiago',
  'cc': 'CL'}]

In [13]:
print(a[0]["cc"])   # AR
print(a[0]["name"]) # Buenos Aires

CL
Villa Presidente Frei, Nunoa, Santiago, Chile


In [1]:
import sys
sys.path.append("../ingestion/python/src")
from APIendpoint import PlacesAPI
sys.path.append("../ingestion/python/data-enrichment-LLM")
from silver_enrichment import *

llm = LLM()
find_mails = FindMails(llm=llm)

test_state = {
    "company_name": "Accenture",
    "city": "Santiago",
    "country": "CL"
}

result = find_mails(test_state)
print(result)


c:\APPS\Python312\Lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{'contacts': [{'email': 'hr.recruitment@accenture.com', 'score': 0.9, 'reason': 'HR/recruitment email'}, {'email': 'karla.hernandez@accenture.com', 'score': 0.85, 'reason': 'HR Service Delivery email'}, {'email': 'contact@accenture.com', 'score': 0.4, 'reason': 'Generic contact email'}]}
